# 🧩 Mini-Lab: Typed Shapes — Hints and Dataclasses

**Module 2: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Annotate** parameters and returns with type hints a checker can verify
2. **Predict** which snippets a type checker flags
3. **Convert** loose dicts into dataclasses, and choose between a dict, a dataclass and a full class

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Type hints | Documentation the machine can check; no runtime behaviour change |
| Dataclass as a shape | Function for behaviour, dataclass for shape, class only for both |


## Why Type Hints Matter for AI Engineering

In the previous mini-lab, we extracted pure functions, with every input as a parameter and every output as a return value. Type hints go one step further. They **document the contract** of each parameter and the return value, so that:

- Your **IDE** auto-completes correctly and warns you about wrong types *as you type*.
- A **type checker** (mypy, Pyright) catches mismatches before you run anything.

Type hints are *optional* at runtime, and Python won't enforce them. Tooling will, though, and that's where the bugs get caught.

Below is `score_documents` from the previous mini-lab, first without hints and then with them.

> 🧩 **Type Hints.** The optional labels from the video, now on a function you know.

In [1]:
# Un-annotated — what types does this expect? What does it return?
def score_documents_untyped(query, documents, threshold=0.3):
    """Score documents by keyword overlap with the query."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

In [2]:
def score_documents(query: str, documents: list[str], threshold: float = 0.3) -> list[dict]:
    """Score documents by keyword overlap with the query."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

In [3]:
# Quick test of the typed version
docs = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

results = score_documents("How do transformers handle long sequences?", docs)
print(results)

[]


In [4]:
def build_request(prompt: str, model: str, max_tokens: int = 500) -> dict[str, object]:
    """Build an LLM API request dict."""
    return {
        "model": model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }


def find_best_chunk(scores: list[float]) -> int | None:
    """Return the index of the highest score, or None if the list is empty."""
    if not scores:
        return None
    return scores.index(max(scores))

In [5]:
print(build_request("Explain RAG", "gpt-4o-mini"))

print(find_best_chunk([0.4, 0.9, 0.7]))   # → 1
print(find_best_chunk([]))                  # → None

{'model': 'gpt-4o-mini', 'max_tokens': 500, 'messages': [{'role': 'user', 'content': 'Explain RAG'}]}
1
None


## How Type Hints Catch Bugs

Type hints don't enforce anything at runtime — Python happily runs mismatched types until something crashes. The value is that **tooling catches the bug earlier**.

Let's see an example of a bug that type hints would flag:

In [6]:
# This call has a bug: threshold should be a float, not a string.
# A type checker flags it BEFORE you run it; without hints it crashes at runtime.

# score_documents("transformers", docs, threshold="high")  # ← TypeError at runtime!

print('Type checker would flag: threshold="high" (expected float)')

Type checker would flag: threshold="high" (expected float)


---
## Dataclasses as shapes

Type hints document the function. A dataclass documents the record.


## The Problem: Dicts Everywhere

In the previous mini-labs, our `score_documents` function returns `list[dict]`. That works — but what keys does each dict have? What types are the values? You have to read the function body to find out:

```python
results.append({"text": doc, "score": round(overlap, 2)})
```

A dict is **shapeless** — Python doesn't stop you from writing `result["txt"]` (typo) or `result["score"] + result["text"]` (type mismatch). You only find out at runtime.

A **dataclass** gives this bundle of values a name and typed fields so tooling catches mistakes before you run anything.

## Dict vs Dataclass: Side by Side

> 🧩 **Dataclass as a shape.** The function-for-behaviour, dataclass-for-shape split from the video, applied here.

In [7]:
# --- Dict approach: no structure enforced ---
result_dict = {"text": "Transformers use attention.", "score": 0.71}

# Typo goes unnoticed until runtime
# result_dict["txt"]  # ← KeyError, but nothing warns you beforehand

print(result_dict["score"])

0.71


In [8]:
from dataclasses import dataclass


@dataclass
class ScoredDoc:
    """A document with its relevance score."""
    text: str
    score: float


# --- Dataclass approach: named, typed fields ---
result_dc = ScoredDoc(text="Transformers use attention.", score=0.71)

# result_dc.txt  # ← IDE underlines this immediately (AttributeError)

print("Dataclass:", result_dc)
print("Access:   ", result_dc.score)

Dataclass: ScoredDoc(text='Transformers use attention.', score=0.71)
Access:    0.71


## Using a Dataclass in a Function

Let's refactor `score_documents` to return `list[ScoredDoc]` instead of `list[dict]`. The logic is identical — only the container changes.

In [9]:
def score_documents(query: str, documents: list[str], threshold: float = 0.3) -> list[ScoredDoc]:
    """Score documents by keyword overlap. Returns ScoredDoc instances."""
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append(ScoredDoc(text=doc, score=round(overlap, 2)))
    results.sort(key=lambda x: x.score, reverse=True)
    return results


# Reuse the docs list defined earlier
results = score_documents("How do transformers handle long sequences?", docs)
if results:
    print(results[0])

Now callers know the exact fields: `.text` and `.score`.

## When Does a Full Class Earn Its Place?

A dataclass is a **shape**, meaning it holds data. A full class adds **behavior**: methods that do non-trivial work, manage internal state, or control a lifecycle like open/close.

Here's the decision rule:

| Situation | Use |
|-----------|-----|
| A bundle of related values passed around | **Dataclass** |
| A throwaway one-off with 1–2 keys | **Dict** (it's fine) |
| Logic that transforms inputs → outputs | **Pure function** |
| An object that manages state or resources (DB connection, client session) | **Full class** |

## Quick Decision Checklist

1. **One-off with 1–2 keys?** → **dict**
2. **Named bundle passed between functions?** → **dataclass**
3. **Manages changing state or resources?** → **full class**
4. **Pure input → output logic?** → **function**

## 🎯 Summary

### Key Takeaways

1. **Type hints annotate the contract.** They declare what a function expects and what it returns, so pure functions document themselves.
2. **Common patterns are simple.** `str`, `list[str]`, `dict[str, object]` and `int | None` cover most AI engineering code.
3. **Bugs get caught earlier.** Your IDE and type checkers (mypy, Pyright) flag type mismatches before runtime, which is far cheaper than debugging a crashed pipeline.
4. **Zero runtime cost.** Python ignores type hints at runtime. They exist purely for tooling and human readers.
5. **Dataclass as a shape.** A named, typed bundle of fields replaces a shapeless dict. Use a full class only when you need state and behaviour together.